# 🛡️ ToxGuard — Multilingual Toxic Comment Classifier

**NeuroLogic '26 | Challenge 3: Multilingual Toxic Comment Classification**

- **Model:** XLM-RoBERTa Base
- **Task:** Binary Toxicity Classification (multilingual)
- **Evaluation Metric:** ROC-AUC
- **Best ROC-AUC:** `0.9900` (Epoch 2)
- **Accuracy:** `95.26%`

---

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install required libraries
!pip install transformers datasets scikit-learn openpyxl accelerate gradio -q

## ⚙️ Cell 2 — Imports & Hardware Check

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import Dataset

# Hardware Check
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Hardware Check - Using device: {DEVICE}')

## 📂 Cell 3 — Load Dataset

In [ ]:
# Update these paths to your actual dataset location
TRAIN_PATH = 'train.xlsx'   # or .csv
TEST_PATH  = 'test.xlsx'    # or .csv
TEXT_COL   = 'text'         # column name containing the comment text
LABEL_COL  = 'toxic'        # column name containing the label (0 or 1)

print('Loading dataset files...')

if TRAIN_PATH.endswith('.xlsx'):
    train_df = pd.read_excel(TRAIN_PATH)
    test_df  = pd.read_excel(TEST_PATH)
else:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df  = pd.read_csv(TEST_PATH)

# Drop rows with missing text or labels
train_df = train_df[[TEXT_COL, LABEL_COL]].dropna()
train_df[TEXT_COL] = train_df[TEXT_COL].astype(str).str.strip()
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)

test_df = test_df[[TEXT_COL]].dropna()
test_df[TEXT_COL] = test_df[TEXT_COL].astype(str).str.strip()

print(f'Train samples : {len(train_df)}')
print(f'Test  samples : {len(test_df)}')
print(f'Label distribution:\n{train_df[LABEL_COL].value_counts()}')
train_df.head()

## ✂️ Cell 4 — Train / Validation Split

In [ ]:
RANDOM_SEED = 42
VAL_SIZE    = 0.15   # 85% train / 15% validation

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df[TEXT_COL].tolist(),
    train_df[LABEL_COL].tolist(),
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_df[LABEL_COL].tolist()
)

print(f'Training samples   : {len(train_texts)}')
print(f'Validation samples : {len(val_texts)}')

## 🧠 Cell 5 — Load XLM-RoBERTa Tokenizer & Model

In [ ]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN    = 128

print('Downloading XLM-RoBERTa Brain (This might take a minute)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
print(f'Model loaded: {MODEL_NAME}')

## 🔢 Cell 6 — Tokenise & Build PyTorch Datasets

In [ ]:
print('Translating words into numbers for the AI...')

class ToxicDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = labels

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ToxicDataset(train_texts, train_labels)
val_dataset   = ToxicDataset(val_texts,   val_labels)
test_dataset  = ToxicDataset(test_df[TEXT_COL].tolist())

print(f'Train dataset size : {len(train_dataset)}')
print(f'Val   dataset size : {len(val_dataset)}')
print(f'Test  dataset size : {len(test_dataset)}')

## 📊 Cell 7 — Custom Metrics (ROC-AUC + Accuracy)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = np.argmax(logits, axis=-1)
    roc   = roc_auc_score(labels, probs)
    acc   = accuracy_score(labels, preds)
    return {'roc_auc': roc, 'accuracy': acc}

## 🚀 Cell 8 — Training Arguments & Trainer

In [ ]:
OUTPUT_DIR = './toxguard_model'

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    learning_rate               = 2e-5,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'roc_auc',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print('Starting the training process... Watch for the ROC-AUC score!')
trainer.train()

## ✅ Cell 9 — Final Evaluation on Validation Set

In [ ]:
metrics = trainer.evaluate()
print('\n===== FINAL EVALUATION METRICS =====')
print(f"  eval_loss     : {metrics['eval_loss']:.4f}")
print(f"  eval_roc_auc  : {metrics['eval_roc_auc']:.4f}")
print(f"  eval_accuracy : {metrics['eval_accuracy']:.4f}")
print('======================================')

## 📤 Cell 10 — Generate Submission CSV

In [ ]:
print('Generating predictions on test set...')

raw_preds = trainer.predict(test_dataset)
probs = torch.softmax(torch.tensor(raw_preds.predictions), dim=-1).numpy()[:, 1]

submission = pd.DataFrame({
    'text'             : test_df[TEXT_COL].tolist(),
    'toxic_probability': probs
})

submission.to_csv('submission.csv', index=False)
print(f'Saved submission.csv with {len(submission)} rows')
submission.head(10)

## 💾 Cell 11 — Save Model & Tokenizer

In [ ]:
SAVE_DIR = './toxguard_final'
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Model and tokenizer saved to: {SAVE_DIR}')

## 🎨 Cell 12 — Gradio Demo

> Run this cell after training. It launches an interactive web app where you can
> type any text in any language and get an instant toxicity prediction.
> `share=True` gives a public URL valid for 72 hours.

In [ ]:
import gradio as gr

# Load saved model (run this section independently if needed)
SAVE_DIR   = './toxguard_final'
_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
_model.eval()
_model.to(DEVICE)

LABELS = [
    (0.0,  0.3,  '✅ Non-Toxic',   '#22c55e'),
    (0.3,  0.6,  '⚠️ Borderline',  '#f59e0b'),
    (0.6,  1.01, '🚨 Toxic',        '#ef4444'),
]

def classify_text(text: str):
    if not text.strip():
        return 'Please enter some text.'
    enc = _tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=128,
        padding=True
    ).to(DEVICE)
    with torch.no_grad():
        logits = _model(**enc).logits
    prob = torch.softmax(logits, dim=-1)[0][1].item()
    verdict, hex_col = '✅ Non-Toxic', '#22c55e'
    for lo, hi, label, color in LABELS:
        if lo <= prob < hi:
            verdict, hex_col = label, color
            break
    html_out = f"""
    <div style='font-family:sans-serif;padding:16px;border-radius:10px;
                background:#1e1e2e;color:#cdd6f4;'>
      <h2 style='margin:0 0 8px;color:{hex_col};'>{verdict}</h2>
      <p style='margin:0;font-size:18px;'>
        Toxicity Probability: <strong style='color:{hex_col};'>{prob:.2%}</strong>
      </p>
      <div style='margin-top:12px;background:#313244;border-radius:6px;height:16px;width:100%;'>
        <div style='height:16px;border-radius:6px;width:{prob*100:.1f}%;background:{hex_col};'></div>
      </div>
    </div>
    """
    return html_out

demo = gr.Interface(
    fn          = classify_text,
    inputs      = gr.Textbox(
                    label='Enter Text (any language)',
                    placeholder='Type a comment in English, Hindi, or any language...',
                    lines=4
                  ),
    outputs     = gr.HTML(label='Result'),
    title       = '🛡️ ToxGuard — Multilingual Toxicity Detector',
    description = (
        'Powered by XLM-RoBERTa fine-tuned on multilingual toxic comments. '
        'Supports 100+ languages. Best Validation ROC-AUC: 0.9900'
    ),
    examples    = [
        ['She is one of the best actresses in Bollywood!'],
        ['I hope she gets what she deserves, stupid bitch.'],
        ['यह एक अच्छा काम है, बधाई हो!'],
        ['California would be a better place without all the dirty mexicans'],
        ['The weather today is really beautiful and sunny.'],
    ],
    theme          = gr.themes.Soft(),
    allow_flagging = 'never'
)

demo.launch(share=True)  # share=True gives a public URL valid for 72 hrs